In [ ]:
import sys
import re
import time
import json
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dateutil import parser as dtparser
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import csv
import hashlib
import html
import os
from datetime import datetime
from pathlib import Path
from typing import List, Optional

In [ ]:
# CNN Scraper
# 1) Use Selenium to paginate CNN "edition" search HTML pages and COLLECT ARTICLE LINKS
# 2) Visit each article link and extract title/date/authors/text from the ARTICLE HTML
# 3) Save results to CSV (links-only + full articles)


# -----------------------------
# 1)
# -----------------------------
def requestPage(link, timeout=25, headers=None):
    try:
        h = headers or {}
        resp = requests.get(link, timeout=timeout, headers=h)
        resp.raise_for_status()
        return resp.text
    except requests.RequestException as e:
        print(f"Error fetching page: {e} -> {link}")
        return None


def requestPageUsingWebDriver(link, headless=True):
    options = Options()
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    options.add_argument(
        "user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(link)
    driver.implicitly_wait(10)
    html = driver.page_source
    driver.quit()
    return html


# -----------------------------
# 2)
# -----------------------------
CNN_BASE = "https://edition.cnn.com"

# Story-like CNN URL patterns
STORY_DATE_RE = re.compile(r"cnn\.com/\d{4}/\d{2}/\d{2}/")
INDEX_RE = re.compile(r"cnn\.com/.+?/index\.html")

BAD_URL_PARTS = [
    "/videos", "/audio", "/profiles", "/interactive", "/specials",
    "/cnn-underscored", "/shopping", "/travel", "/style"
]

def is_cnn_story_url(url: str) -> bool:
    if not url or "cnn.com" not in url:
        return False
    if any(b in url for b in BAD_URL_PARTS):
        return False
    return bool(STORY_DATE_RE.search(url) or INDEX_RE.search(url))

def build_cnn_search_url(query: str, page: int, size: int = 10, sort: str = "newest"):
    offset = (page - 1) * size
    return (f"{CNN_BASE}/search?q={query}"
            f"&from={offset}&size={size}&page={page}&sort={sort}&types=all&section=")

def extract_links_from_rendered_search_html(html: str):
    soup = BeautifulSoup(html, "lxml")
    urls = []
    for a in soup.select("a[href]"):
        href = a.get("href")
        if not href:
            continue
        # Make absolute
        if href.startswith("/"):
            href = "https://www.cnn.com" + href  # results often point to www.cnn.com
        # Some are already absolute:
        if href.startswith("http") and is_cnn_story_url(href):
            urls.append(href)

    # De-dupe preserving order
    seen = set()
    out = []
    for u in urls:
        if u in seen:
            continue
        seen.add(u)
        out.append(u)
    return out

def extractCNNArticleLinksFromSearch(query: str,
                                     max_pages: int = 50,
                                     size: int = 10,
                                     sort: str = "newest",
                                     headless: bool = True,
                                     min_new_urls_to_continue: int = 1):
    """
    Uses Selenium to render the CNN search page HTML.
    Then extracts story-like URLs by scanning <a href> in the rendered HTML.
    Stops when a page yields no NEW urls.
    """
    options = Options()
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1400,900")
    options.add_argument(
        "user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    wait = WebDriverWait(driver, 25)

    all_urls = []
    seen = set()

    try:
        for page in range(1, max_pages + 1):
            search_url = build_cnn_search_url(query=query, page=page, size=size, sort=sort)
            print(f"[SEARCH PAGE {page}] {search_url}")
            driver.get(search_url)

            # Best-effort wait: wait until there are some anchors (page is interactive)
            try:
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href]")))
                time.sleep(2.0)  # allow JS results to populate
            except Exception:
                pass

            html = driver.page_source
            urls = extract_links_from_rendered_search_html(html)

            # Add only new URLs
            new_urls = [u for u in urls if u not in seen]
            for u in new_urls:
                seen.add(u)
                all_urls.append(u)

            print(f"  -> found {len(urls)} urls ({len(new_urls)} new)")

            # Stop when no new links
            if len(new_urls) < min_new_urls_to_continue:
                print("  -> no new urls, stopping pagination")
                break

            time.sleep(random.uniform(0.8, 1.6))

    finally:
        driver.quit()

    return all_urls


# -----------------------------
# 3) (Optional) Scrape each CNN article HTML for title/date/authors/text
# -----------------------------
ARTICLE_BODY_SELECTORS = [
    "section[data-component-name='article-body']",
    "div[data-component-name='article-body']",
    "div.article__content",
    "div.article__body",
    "main article",
    "article",
]

def parse_jsonld(soup: BeautifulSoup):
    blocks = soup.find_all("script", attrs={"type": "application/ld+json"})
    for b in blocks:
        txt = b.get_text(strip=True)
        if not txt:
            continue
        try:
            data = json.loads(txt)
        except Exception:
            continue

        candidates = []
        if isinstance(data, dict):
            candidates = [data]
        elif isinstance(data, list):
            candidates = [d for d in data if isinstance(d, dict)]

        for d in candidates:
            t = d.get("@type")
            t_str = " ".join(t) if isinstance(t, list) else str(t or "")
            if "Article" in t_str or "NewsArticle" in t_str or "ReportageNewsArticle" in t_str:
                return d
    return {}

def normalize_dt(v):
    if not v:
        return None
    try:
        dt = dtparser.parse(v)
        if not dt.tzinfo:
            # if timezone missing, treat as UTC
            dt = dt.replace(tzinfo=None)
        return dt.isoformat()
    except Exception:
        return None

def scrape_cnn_article(url: str, timeout=25):
    headers = {
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    html = requestPage(url, timeout=timeout, headers=headers)
    if not html:
        return None

    soup = BeautifulSoup(html, "lxml")

    # canonical
    canonical = None
    canon_tag = soup.find("link", rel="canonical")
    if canon_tag and canon_tag.get("href"):
        canonical = canon_tag["href"].strip()

    meta = parse_jsonld(soup)
    title = meta.get("headline") or meta.get("name")
    published = normalize_dt(meta.get("datePublished") or meta.get("dateCreated") or meta.get("dateModified"))
    section = meta.get("articleSection") if isinstance(meta.get("articleSection"), str) else None

    # authors
    authors = None
    author_field = meta.get("author")
    if author_field:
        if isinstance(author_field, list):
            authors = []
            for a in author_field:
                if isinstance(a, dict) and "name" in a:
                    authors.append(a["name"])
                elif isinstance(a, str):
                    authors.append(a)
            authors = authors or None
        elif isinstance(author_field, dict) and "name" in author_field:
            authors = [author_field["name"]]
        elif isinstance(author_field, str):
            authors = [author_field]

    # text: find container then join paragraphs
    container = None
    for sel in ARTICLE_BODY_SELECTORS:
        container = soup.select_one(sel)
        if container:
            break

    text = None
    if container:
        paras = []
        for p in container.find_all("p"):
            t = p.get_text(" ", strip=True)
            if t and len(t) >= 25:
                paras.append(t)
        joined = "\n\n".join(paras).strip()
        if len(joined) >= 300:
            text = joined

    return {
        "url": url,
        "canonical_url": canonical,
        "title": " ".join(title.split()) if isinstance(title, str) else None,
        "published": published,
        "authors": authors,
        "section": section,
        "text": text,
    }


def scrape_articles(urls: List[str], keyword: str = "gaza", max_articles: int = 500):
    rows = []
    seen = set()
    for i, url in enumerate(urls, 1):
        if len(rows) >= max_articles:
            break
        time.sleep(random.uniform(0.8, 1.6))

        art = scrape_cnn_article(url)
        if not art:
            continue

        canon = art.get("canonical_url") or art["url"]
        if canon in seen:
            continue
        seen.add(canon)

        blob = " ".join([(art.get("title") or ""), (art.get("text") or "")]).lower()
        if keyword.lower() not in blob:
            continue

        # keep even if text is None? up to you:
        if not art.get("text"):
            continue

        rows.append(art)
        print(f"[OK] {len(rows):04d} {art.get('published')} {art.get('title')}")

    return rows


# -----------------------------
# 4) RUN
# -----------------------------
query = "hamas"
max_pages = 200
size = 10

# Step A: collect links from CNN search pages
links = extractCNNArticleLinksFromSearch(
    query=query,
    max_pages=max_pages,
    size=size,
    sort="newest",
    headless=True
)

print("Total links:", len(links))
print("Last link:", links[-1] if links else None)

# Save links-only
df_links = pd.DataFrame({"url": sorted(set(links))})
df_links.to_csv("cnn_links_hamas.csv", index=False)
print("Saved links to cnn_links_hamas.csv")

# Step B (optional): scrape full articles
articles = scrape_articles(df_links["url"].tolist(), keyword="gaza", max_articles=1000)

df_articles = pd.DataFrame(articles)
df_articles.to_csv("cnn_hamas_articles.csv", index=False)
print("Saved full articles to cnn_hamas_articles.csv")


[SEARCH PAGE 1] https://edition.cnn.com/search?q=hamas&from=0&size=10&page=1&sort=newest&types=all&section=
  -> found 10 urls (10 new)
[SEARCH PAGE 2] https://edition.cnn.com/search?q=hamas&from=10&size=10&page=2&sort=newest&types=all&section=
  -> found 8 urls (6 new)
[SEARCH PAGE 3] https://edition.cnn.com/search?q=hamas&from=20&size=10&page=3&sort=newest&types=all&section=
  -> found 12 urls (10 new)
[SEARCH PAGE 4] https://edition.cnn.com/search?q=hamas&from=30&size=10&page=4&sort=newest&types=all&section=
  -> found 11 urls (9 new)
[SEARCH PAGE 5] https://edition.cnn.com/search?q=hamas&from=40&size=10&page=5&sort=newest&types=all&section=
  -> found 11 urls (9 new)
[SEARCH PAGE 6] https://edition.cnn.com/search?q=hamas&from=50&size=10&page=6&sort=newest&types=all&section=
  -> found 12 urls (10 new)
[SEARCH PAGE 7] https://edition.cnn.com/search?q=hamas&from=60&size=10&page=7&sort=newest&types=all&section=
  -> found 10 urls (8 new)
[SEARCH PAGE 8] https://edition.cnn.com/search?

[OK] 0041 2025-11-17T22:17:35.211000+00:00 Trump’s plan for Gaza endorsed by UN Security Council
[OK] 0042 2025-11-18T22:59:42.183000+00:00 Israeli strike on a Palestinian refugee camp in Lebanon kills 13 people, Lebanon’s health ministry says
[OK] 0043 2025-11-19T21:59:40.971000+00:00 Israeli strikes across Gaza kill at least 32 Palestinians, as Hamas warns of ‘dangerous escalation’
[OK] 0044 2025-11-20T16:47:54.198000+00:00 Israel’s forced expulsion of Palestinians from West Bank refugee camps amounts to war crimes, HRW alleges
[OK] 0045 2025-11-21T12:00:59.627000+00:00 Trump administration rushes to emulate Gaza plan in new push for Ukraine Deal
[OK] 0046 2025-11-24T15:18:16.489000+00:00 Controversial Gaza Humanitarian Foundation to close aid effort
[OK] 0047 2025-11-26T07:36:32.125000+00:00 Israel identifies deceased hostage returned by Hamas as Dror Or. Two more hostage bodies remain in Gaza
[OK] 0048 2025-12-03T15:55:08.404000+00:00 Far right coalition boycotts Knesset vote backi

In [ ]:
"""
Merge multiple CNN article CSVs, deduplicate, normalize dates, and clean article text
"""

# ---------- CONFIG ----------
FILES = [
    "cnn_gaza_articles.csv",
    "cnn_hamas_articles.csv",
    "cnn_palestine_articles.csv",
    "cnn_idf_articles.csv",
    "cnn_israel_articles.csv",
]

OUT_CSV = "cnn_merged_articles.csv"
OUT_JSONL = "cnn_merged_articles.jsonl"

# Minimum paragraph length to keep (after cleaning). Paragraphs shorter than this are pruned.
MIN_PARAGRAPH_LEN = 30

# Patterns / tokens to treat as boilerplate and remove where appropriate
BOILERPLATE_PATTERNS = [
    r"Read more on cnn",
    r"Read more:", r"Related:", r"Click here", r"Follow the author",
    r"Sign up to.*", r"Follow .* on", r"Subscribe to", r"More on .*",
]
BOILERPLATE_RE = re.compile("|".join([f"(?:{p})" for p in BOILERPLATE_PATTERNS]), flags=re.I)

# ---------- UTILITIES ----------
def load_csvs(files: List[str]) -> pd.DataFrame:
    dfs = []
    for f in files:
        if not Path(f).exists():
            print(f"[WARN] file not found, skipping: {f}")
            continue
        df = pd.read_csv(f, dtype=str, keep_default_na=False)  # load as strings
        print(f"[INFO] loaded {len(df):,} rows from {f}")
        dfs.append(df)
    if not dfs:
        raise SystemExit("No input files found. Place CSVs in the working directory or update FILES.")
    combined = pd.concat(dfs, ignore_index=True, sort=False)
    print(f"[INFO] combined total rows: {len(combined):,}")
    return combined


def canonicalize_url(u: Optional[str]) -> Optional[str]:
    if not u or not isinstance(u, str) or u.strip() == "":
        return None
    u = u.strip()
    # Normalize scheme-less and relative (best-effort)
    if u.startswith("//"):
        u = "https:" + u
    # Lower-case scheme & host
    try:
        from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

        p = urlparse(u)
        scheme = p.scheme or "https"
        netloc = p.netloc.lower()
        path = p.path or "/"
        # Remove trailing slash except root
        if path != "/" and path.endswith("/"):
            path = path[:-1]
        # Filter out common tracking/query params
        query_pairs = [(k, v) for k, v in parse_qsl(p.query, keep_blank_values=True) if not k.lower().startswith("utm_")]
        query = urlencode(query_pairs, doseq=True)
        normalized = urlunparse((scheme, netloc, path, "", query, ""))
        return normalized
    except Exception:
        # fallback: strip whitespace and trailing slash
        u = u.split("?")[0]
        if u.endswith("/"):
            u = u[:-1]
        return u


def parse_datetime(v: Optional[str]) -> Optional[datetime]:
    if not v or (isinstance(v, float) and pd.isna(v)):
        return None
    v = str(v).strip()
    if v == "":
        return None
    # try parsing with dateutil
    try:
        dt = dtparser.parse(v)
        return dt
    except Exception:
        # last attempt: common CNN format like "Feb 7, 2026"
        try:
            return datetime.strptime(v, "%b %d, %Y")
        except Exception:
            return None


def clean_html_text(raw: Optional[str]) -> Optional[str]:
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return None
    txt = str(raw)

    # use BeautifulSoup to strip tags; otherwise unescape & normalize
    # Remove common HTML comments first
    txt = re.sub(r"<!--.*?-->", " ", txt, flags=re.S)

    # If we see HTML tags (heuristic), parse them; else treat as plain text
    if "<" in txt and ">" in txt:
        soup = BeautifulSoup(txt, "lxml")
        # remove scripts/styles
        for s in soup(["script", "style", "aside", "figure", "footer", "nav"]):
            s.decompose()
        # remove elements that are likely to be metadata/advertising
        for sel in soup.select(".ad, .advert, .promo, .share-tools, .related"):
            sel.decompose()
        # get text
        txt = soup.get_text("\n")
    else:
        # plain text: keep as is
        pass

    # HTML-unescape entities
    txt = html.unescape(txt)

    # Normalize newlines/whitespace
    # split into lines, trim, remove repeated lines and short boilerplate lines
    lines = [re.sub(r"\s+", " ", line).strip() for line in txt.splitlines()]
    cleaned_lines = []
    for line in lines:
        if not line:
            continue
        # remove boilerplate-like lines
        if BOILERPLATE_RE.search(line):
            continue
        cleaned_lines.append(line)

    # Join lines into paragraphs: collapse sequences of short lines into paragraphs
    # Merge consecutive lines into paragraphs when lines are short
    paragraphs = []
    buf = []
    for line in cleaned_lines:
        buf.append(line)
        # if current line ends with sentence end or is long, flush buffer to a paragraph
        if len(line) > 120 or line.endswith((".", "?", "!")):
            paragraphs.append(" ".join(buf))
            buf = []
    if buf:
        paragraphs.append(" ".join(buf))

    # Further prune very short paragraphs (likely captions / noise)
    paragraphs = [p.strip() for p in paragraphs if len(p.strip()) >= MIN_PARAGRAPH_LEN]

    # Final join with double newlines (good for LLM reading)
    final = "\n\n".join(paragraphs).strip()

    # As a final safety: collapse more than 3 newlines, strip leading/trailing
    final = re.sub(r"\n{3,}", "\n\n", final).strip()

    return final or None


def text_fingerprint(text: Optional[str]) -> Optional[str]:
    if not text:
        return None
    h = hashlib.sha256()
    # normalize whitespace
    normalized = re.sub(r"\s+", " ", text).strip()
    h.update(normalized.encode("utf-8'))
    return h.hexdigest()


# ---------- MAIN PROCESS ----------
def main():
    df = load_csvs(FILES)

    # Ensure columns exist; rename if necessary
    # unify to: url, title, text, date
    col_map = {}
    cols = set(df.columns.str.lower())
    # find likely date column
    for candidate in ["date", "published", "published_at", "scraped_at", "created"]:
        for c in df.columns:
            if c.lower() == candidate:
                col_map[c] = "date"
    # find title/text/url if uppercase or different names
    for c in df.columns:
        if c.lower() == "title":
            col_map[c] = "title"
        if c.lower() in ("text", "article", "body"):
            col_map[c] = "text"
        if c.lower() in ("url", "link", "article_url"):
            col_map[c] = "url"
        if c.lower() in ("canonical_url", "canonical"):
            col_map[c] = "canonical_url"
    # apply rename
    if col_map:
        df = df.rename(columns=col_map)

    # ensure necessary columns exist
    for needed in ("url", "title", "text"):
        if needed not in df.columns:
            df[needed] = None

    # keep canonical_url if present
    has_canon = "canonical_url" in df.columns

    # select only the columns we want + canonical if present
    keep_cols = ["url", "title", "text", "date"]
    if has_canon:
        keep_cols.append("canonical_url")
    df = df.reindex(columns=[c for c in keep_cols if c in df.columns])

    # canonicalize urls
    df["url_norm"] = df["url"].apply(canonicalize_url)
    if has_canon:
        df["canonical_norm"] = df["canonical_url"].apply(canonicalize_url)
    else:
        df["canonical_norm"] = None

    # parse dates
    df["date_parsed"] = df["date"].apply(parse_datetime)

    # clean text
    df["text_clean"] = df["text"].apply(clean_html_text)

    # compute text fingerprint
    df["text_hash"] = df["text_clean"].apply(text_fingerprint)

    # duplicate removal strategy:
    # 1) prefer canonical_norm (if not null)
    # 2) else url_norm
    # 3) else text_hash
    dedupe_key = []
    final_records = []
    seen_keys = set()
    total_rows = len(df)

    # We'll iterate rows in original order and keep first occurrence
    for idx, row in df.iterrows():
        key = None
        if row.get("canonical_norm"):
            key = ("canon", row["canonical_norm"])
        elif row.get("url_norm"):
            key = ("url", row["url_norm"])
        elif row.get("text_hash"):
            key = ("text", row["text_hash"])
        else:
            # fallback: title + first 100 chars of text
            title = (row.get("title") or "").strip()
            short = (row.get("text_clean") or "")[:100]
            key = ("titletext", hashlib.sha256((title + short).encode("utf-8")).hexdigest())

        if key in seen_keys:
            continue
        seen_keys.add(key)

        # construct record with requested columns
        rec = {
            "url": row.get("url_norm") or row.get("url"),
            "title": (row.get("title") or "").strip() or None,
            "text": row.get("text_clean"),
            "date": row.get("date_parsed"),
        }
        final_records.append(rec)

    merged_df = pd.DataFrame(final_records)
    print(f"[INFO] initial rows: {total_rows:,}, unique articles after dedupe: {len(merged_df):,}")

    # Evaluation: missing text / short text stats
    missing_text = merged_df["text"].isna().sum()
    short_text = (merged_df["text"].fillna("").str.len() < 300).sum()
    date_missing = merged_df["date"].isna().sum()

    print("=== QUALITY REPORT ===")
    print(f"Total merged articles: {len(merged_df):,}")
    print(f"Missing text: {missing_text:,} ({missing_text/len(merged_df):.1%})")
    print(f"Short text (<300 chars): {short_text:,} ({short_text/len(merged_df):.1%})")
    print(f"Missing/unknown date: {date_missing:,} ({date_missing/len(merged_df):.1%})")

    # Heuristic advice
    if missing_text > 0:
        print("\n[ADVICE] Some articles are missing extracted text.")
        print("  - Consider re-scraping those URLs with a different article-body selector")
        print("  - Or open the HTML debug files if you saved them during scraping to inspect DOM")
    if short_text / max(1, len(merged_df)) > 0.3:
        print("\n[ADVICE] >30% of articles have short text. Consider:")
        print("  - relaxing MIN_PARAGRAPH_LEN or the minimum text length threshold")
        print("  - using a fallback extractor (newspaper3k or trafilatura) for those pages")

    # Save to CSV and JSONL
    # Convert date to ISO for CSV
    merged_df["date_iso"] = merged_df["date"].apply(lambda d: d.isoformat() if pd.notna(d) else "")
    out_cols = ["url", "title", "text", "date_iso"]
    merged_df.to_csv(OUT_CSV, index=False, columns=out_cols)

    # JSONL with datetime as ISO
    with open(OUT_JSONL, "w", encoding="utf-8") as fo:
        for _, r in merged_df.iterrows():
            obj = {
                "url": r["url"],
                "title": r["title"],
                "text": r["text"],
                "date": r["date"].isoformat() if pd.notna(r["date"]) else None,
            }
            fo.write(json.dumps(obj, ensure_ascii=False) + "\n")

    print(f"\nSaved merged CSV -> {OUT_CSV}")
    print(f"Saved merged JSONL -> {OUT_JSONL}")
    return merged_df


if __name__ == "__main__":
    merged = main()
